# LoRA Fine-Tuning Assignment — Solution

## Objective

Apply what you learned in the SFT + LoRA tutorial to fine-tune **TinyLlama-1.1B** on the **databricks-dolly-15k** dataset using LoRA.

## Prerequisites

- You have completed the SFT + LoRA tutorial notebook.
- Runtime: **GPU** (T4 is sufficient).

---
## Part 0 — Setup (provided)

In [ ]:
!pip install -U transformers trl peft datasets accelerate torch -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 106.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.8/526.8 kB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 55.2 MB/s eta 0:00:00


In [ ]:
import torch
import gc
import os
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, PeftModel, get_peft_model
from trl import SFTConfig, SFTTrainer

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA A100-SXM4-40GB


### Utility functions (provided)

In [ ]:
def clear_memory():
    """Free GPU memory between experiments."""
    gc.collect()
    torch.cuda.empty_cache()
    if torch.cuda.is_available():
        print(f"GPU allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
        print(f"GPU reserved:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB")


def print_trainable_params(model):
    """Display trainable vs total parameter counts."""
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"Trainable parameters: {trainable:>12,}  ({100 * trainable / total:.4f}%)")
    print(f"Total parameters:     {total:>12,}")


def generate_response(model, tokenizer, prompt, max_new_tokens=256):
    """Generate a response from a model given a prompt string."""
    messages = [{"role": "user", "content": prompt}]
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return response

---
## Part 1 — Configuration

### Task 1: Define all hyperparameters

In [ ]:
# ── SOLUTION ──
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
DATASET_NAME = "databricks/databricks-dolly-15k"
NUM_TRAIN_SAMPLES = 1000
MAX_SEQ_LENGTH = 512

# LoRA hyperparameters
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj", "v_proj"]

# Training hyperparameters
LEARNING_RATE = 2e-4
NUM_EPOCHS = 1
BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 4
OUTPUT_DIR = "./sft-lora-tinyllama"

---
## Part 2 — Load Model and Tokenizer

### Task 2: Load the tokenizer and base model

In [ ]:
# ── SOLUTION ──
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model: {MODEL_NAME}")
print(f"Total parameters: {total_params:,}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model: TinyLlama/TinyLlama-1.1B-Chat-v1.0
Total parameters: 1,100,048,384


---
## Part 3 — Test Base Model

### Task 3: Generate responses from the base model

In [ ]:
# ── SOLUTION ──
test_prompts = [
    "Explain what a neural network is in 2-3 sentences.",
    "What are three tips for better sleep?",
]

print("=" * 60)
print("BASE MODEL RESPONSES (before fine-tuning)")
print("=" * 60)

base_responses = []
for i, prompt in enumerate(test_prompts):
    print(f"\nPrompt {i+1}: {prompt}")
    print("-" * 40)
    response = generate_response(model, tokenizer, prompt, max_new_tokens=200)
    base_responses.append(response)
    print(f"Response: {response[:500]}")

BASE MODEL RESPONSES (before fine-tuning)

Prompt 1: Explain what a neural network is in 2-3 sentences.
----------------------------------------


Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Response: A neural network is a type of artificial intelligence (AI) system that mimics the functioning of the human brain. It consists of a set of interconnected nodes (neurons) that process information based on input data. Neural networks can be trained using a supervised or unsupervised learning approach to learn patterns and relationships in data. They can be used in various applications such as image recognition, natural language processing, and machine learning.

Prompt 2: What are three tips for better sleep?
----------------------------------------
Response: Sure, here are three tips for better sleep:

1. Create a sleep-friendly environment: Make sure your bedroom is cool, dark, and quiet. This helps you to fall asleep faster and stay asleep longer.

2. Limit caffeine and alcohol: Caffeine and alcohol can interfere with sleep, so try to avoid them before bedtime.

3. Establish a sleep routine: A consistent sleep routine can help you fall asleep faster and stay asleep longer. Tr

---
## Part 4 — Prepare the Dataset

### Task 4: Load and format the Dolly dataset

In [ ]:
# ── SOLUTION ──
raw_dataset = load_dataset(DATASET_NAME, split="train")
print(f"Full dataset size: {len(raw_dataset):,} examples")
print(f"Columns: {raw_dataset.column_names}")
print(f"\nExample entry:")
print(raw_dataset[0])


def format_dolly_to_chat(example):
    """Convert Dolly format to conversational messages format."""
    user_content = example["instruction"]
    if example.get("context") and example["context"].strip():
        user_content = f"{example['instruction']}\n\n{example['context']}"

    return {
        "messages": [
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": example["response"]},
        ]
    }


dataset = raw_dataset.shuffle(seed=42).select(range(NUM_TRAIN_SAMPLES))
dataset = dataset.map(
    format_dolly_to_chat,
    remove_columns=raw_dataset.column_names,
    desc="Formatting to chat messages",
)

print(f"\nFormatted dataset size: {len(dataset):,}")
print(f"Columns: {dataset.column_names}")
print(f"\nExample formatted entry:")
print(dataset[0])

README.md: 0.00B [00:00, ?B/s]

databricks-dolly-15k.jsonl:   0%|          | 0.00/13.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

Full dataset size: 15,011 examples
Columns: ['instruction', 'context', 'response', 'category']

Example entry:
{'instruction': 'When did Virgin Australia start operating?', 'context': "Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney.", 'response': 'Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.', 'category': 'closed_qa'}


Formatting to chat messages:   0%|          | 0/1000 [00:00<?, ? examples/s]


Formatted dataset size: 1,000
Columns: ['messages']

Example formatted entry:
{'messages': [{'content': 'Who were the children of the legendary Garth Greenhand, the High King of the First Men in the series A Song of Ice and Fire?', 'role': 'user'}, {'content': 'Garth the Gardener, John the Oak, Gilbert of the Vines, Brandon of the Bloody Blade, Foss the Archer, Owen Oakenshield, Harlon the Hunter, Herndon of the Horn, Bors the Breaker, Florys the Fox, Maris the Maid, Rose of the Red Lake, Ellyn Ever Sweet, Rowan Gold-Tree', 'role': 'assistant'}]}


---
## Part 5 — Configure LoRA

### Task 5: Create the LoRA config and inspect trainable parameters

In [ ]:
# ── SOLUTION ──
peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=LORA_TARGET_MODULES,
)

lora_model = get_peft_model(model, peft_config)

print("LoRA Configuration:")
print(f"  Rank (r):          {peft_config.r}")
print(f"  Alpha:             {peft_config.lora_alpha}")
print(f"  Effective scaling: {peft_config.lora_alpha / peft_config.r}")
print(f"  Target modules:    {peft_config.target_modules}")
print()
print_trainable_params(lora_model)

print("\nLoRA module example (layer 0, q_proj):")
print(lora_model.model.model.layers[0].self_attn.q_proj)

LoRA Configuration:
  Rank (r):          8
  Alpha:             16
  Effective scaling: 2.0
  Target modules:    {'q_proj', 'v_proj'}

Trainable parameters:    1,126,400  (0.1023%)
Total parameters:     1,101,174,784

LoRA module example (layer 0, q_proj):
lora.Linear(
  (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
  (lora_dropout): ModuleDict(
    (default): Dropout(p=0.05, inplace=False)
  )
  (lora_A): ModuleDict(
    (default): Linear(in_features=2048, out_features=8, bias=False)
  )
  (lora_B): ModuleDict(
    (default): Linear(in_features=8, out_features=2048, bias=False)
  )
  (lora_embedding_A): ParameterDict()
  (lora_embedding_B): ParameterDict()
  (lora_magnitude_vector): ModuleDict()
)


---
## Part 6 — Train

### Task 6: Set up SFTTrainer and run training

In [ ]:
# ── SOLUTION ──
# Clean up and reload base model
del lora_model
clear_memory()

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

# Training configuration
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    max_length=MAX_SEQ_LENGTH,
    logging_steps=10,
    logging_first_step=True,
    save_strategy="no",
    seed=42,
)

# Create trainer
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
    peft_config=peft_config,
)

print_trainable_params(trainer.model)

# Train
print("\nStarting training...")
train_result = trainer.train()

metrics = train_result.metrics
print(f"\nTraining complete.")
print(f"  Final loss: {metrics.get('train_loss', 0):.4f}")
print(f"  Runtime:    {metrics.get('train_runtime', 0):.1f}s")

GPU allocated: 2.06 GB
GPU reserved:  2.10 GB


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Tokenizing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (4118 > 2048). Running this sequence through the model will result in indexing errors


Truncating train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Trainable parameters:    1,126,400  (0.1023%)
Total parameters:     1,101,174,784

Starting training...


Step,Training Loss
1,2.202904
10,2.090365
20,1.871092
30,1.748978
40,1.724675
50,1.769201
60,1.712686



Training complete.
  Final loss: 1.8205
  Runtime:    57.0s


---
## Part 7 — Save the Adapter

### Task 7: Save the LoRA adapter and print its size

In [ ]:
# ── SOLUTION ──
adapter_path = f"{OUTPUT_DIR}/final-adapter"
trainer.save_model(adapter_path)
tokenizer.save_pretrained(adapter_path)

adapter_size = sum(
    os.path.getsize(os.path.join(adapter_path, f))
    for f in os.listdir(adapter_path)
    if os.path.isfile(os.path.join(adapter_path, f))
)
print(f"Adapter saved to: {adapter_path}")
print(f"Adapter size: {adapter_size / 1024**2:.1f} MB")

Adapter saved to: ./sft-lora-tinyllama/final-adapter
Adapter size: 7.8 MB


---
## Part 8 — Evaluate

### Task 8: Compare base vs fine-tuned responses

In [ ]:
# ── SOLUTION ──
finetuned_model = trainer.model
finetuned_model.eval()

print("=" * 60)
print("COMPARISON: BASE vs FINE-TUNED")
print("=" * 60)

for i, prompt in enumerate(test_prompts):
    print(f"\n{'━' * 60}")
    print(f"Prompt {i+1}: {prompt}")
    print(f"{'━' * 60}")

    ft_response = generate_response(finetuned_model, tokenizer, prompt, max_new_tokens=200)

    print(f"\n  BASE MODEL:")
    print(f"  {base_responses[i][:300]}")

    print(f"\n  FINE-TUNED (LoRA):")
    print(f"  {ft_response[:300]}")

Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


COMPARISON: BASE vs FINE-TUNED

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Prompt 1: Explain what a neural network is in 2-3 sentences.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



  BASE MODEL:
  A neural network is a type of artificial intelligence (AI) system that mimics the functioning of the human brain. It consists of a set of interconnected nodes (neurons) that process information based on input data. Neural networks can be trained using a supervised or unsupervised learning approach t

  FINE-TUNED (LoRA):
  A neural network is a type of machine learning algorithm that is designed to mimic the behavior of the human brain. It is used in various applications such as image recognition, natural language processing, and speech recognition.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Prompt 2: What are three tips for better sleep?
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  BASE MODEL:
  Sure, here are three tips for better sleep:

1. Create a sleep-friendly environment: Make sure your bedroom is cool, dark, and quiet. This helps you to fall asleep faster and stay asleep longer.

2. Limit caffeine and alcohol: Caffeine and

---
## Done

You have successfully fine-tuned TinyLlama with LoRA on the Dolly dataset. Review your base vs fine-tuned outputs and note the differences in instruction-following quality.